# Diabetic retinopathy screening

Exercise notebook for the lesson **[Diabetic retinopathy screening](https://ramkumarmd.com/medical-ai/applied-diabetes/diabetic-retinopathy-screening)** in the *Applied case studies: diabetes & metabolic health* track of the Medical AI course by Dr. Ram Kumar G.

**In short:** Diabetic retinopathy screening is one of the most established uses of medical AI: systems grade retinal photographs and refer people with referable disease to eye specialists. Deploying one is about more than accuracy: the operating threshold sets how many people are referred and how many cases are missed, ungradable images need another route, and a change of camera can shift performance. Real-world studies have found wide differences between commercial systems, so each programme needs local validation and monitoring.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Simulating a screening programme

A synthetic screening population, threshold selection and programme-level results.

In [ ]:
import numpy as np

rng = np.random.default_rng(3)
GRADES = ["none", "mild", "moderate", "severe", "proliferative"]
PREVALENCE = [0.70, 0.18, 0.08, 0.025, 0.015]     # an illustrative screening population
REFERABLE = {"moderate", "severe", "proliferative"}
SCORE_MEAN = {"none": -1.5, "mild": -0.6, "moderate": 1.0, "severe": 1.8, "proliferative": 2.2}

def screening_population(n, camera_shift=0.0):
    grades = rng.choice(GRADES, n, p=PREVALENCE)
    scores = np.array([rng.normal(SCORE_MEAN[g], 1.0) for g in grades]) + camera_shift
    ungradable = rng.random(n) < 0.05               # poor images: cataract, small pupils, blur
    return grades, scores, ungradable

def summarise(grades, scores, ungradable, threshold, label):
    gradable = ~ungradable
    referable = np.isin(grades, list(REFERABLE))
    flagged = scores >= threshold
    sens = np.mean(flagged[gradable & referable])
    spec = np.mean(~flagged[gradable & ~referable])
    ppv = np.mean(referable[gradable & flagged])
    per_1000 = 1000 / len(grades)
    print(f"{label}: sensitivity {sens:.1%}, specificity {spec:.1%}, PPV {ppv:.1%}")
    print(f"  per 1,000 screened: {np.sum(gradable & flagged) * per_1000:.0f} referred, "
          f"{np.sum(gradable & referable & ~flagged) * per_1000:.1f} referable cases missed, "
          f"{np.sum(ungradable) * per_1000:.0f} ungradable (need another route)")

# Choose the threshold on a development set so that sensitivity is at least 90%
dev_g, dev_s, dev_u = screening_population(20000)
ref_scores = dev_s[~dev_u & np.isin(dev_g, list(REFERABLE))]
threshold = np.quantile(ref_scores, 0.10)
print(f"Operating threshold chosen for 90% sensitivity: {threshold:.2f}\n")
summarise(*screening_population(20000), threshold, "Validation, same camera")

# Operating threshold chosen for 90% sensitivity: -0.04
#
# Validation, same camera: sensitivity 88.3%, specificity 88.8%, PPV 51.7%
#   per 1,000 screened: 194 referred, 13.2 referable cases missed, 49 ungradable (need another route)

### A new camera

Continuing from the previous block: the same threshold with a camera that lowers scores.

In [ ]:
summarise(*screening_population(20000, camera_shift=-0.5), threshold, "New camera, scores 0.5 lower")

# New camera, scores 0.5 lower: sensitivity 78.9%, specificity 95.3%, PPV 70.1%
#   per 1,000 screened: 131 referred, 24.5 referable cases missed, 50 ungradable (need another route)

## Exercise

**Goal:** Plan the operating point and monitoring for an AI retinopathy screening programme.

**Data:** Synthetic screening population in the notebook; optional open retinal image datasets (for example MedMNIST RetinaMNIST) (Generated in the notebook; MedMNIST under CC BY 4.0)

### Step 1

Plot sensitivity, specificity and referrals per 1,000 across thresholds, and mark the 90% sensitivity point.

In [ ]:
# Your code here


### Step 2

Bootstrap the development set to put a confidence interval on the chosen threshold's sensitivity, and choose a safer threshold.

In [ ]:
# Your code here


### Step 3

Model a referral capacity limit (for example 120 per 1,000) and find the best sensitivity achievable within it.

In [ ]:
# Your code here


### Step 4

Design a monthly monitoring report that would have detected the camera change within three months.

In [ ]:
# Your code here


### Step 5

Optional: train the CNN from the imaging track on RetinaMNIST and compare its score distributions with the simulation.

In [ ]:
# Your code here


---

Lesson: [https://ramkumarmd.com/medical-ai/applied-diabetes/diabetic-retinopathy-screening](https://ramkumarmd.com/medical-ai/applied-diabetes/diabetic-retinopathy-screening). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.